# PlasticWatch — Train YOLO11s on Plastic Waste + background photos

Trains the detector on the **Plastic Waste Dataset** (Roboflow Universe, 12,484 images,
CC BY 4.0) **plus background photos that contain no plastic** — streets, parks, sky, rooms —
taken from COCO val2017 with every plastic-looking object class filtered out.

Why the background photos: a model trained only on pictures *of* plastic has never been
told what "no plastic" looks like, so it fires on clean pavements, sky and water. Ultralytics
treats an image with an empty label file as a pure negative: every box it predicts there is
a false positive, and training pushes those down.

Checkpoints are written to Google Drive every 5 epochs, so a Colab disconnect loses at most
a few epochs — just re-run the notebook and training resumes.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Add your Roboflow key under
🔑 *Secrets* (left sidebar) as `ROBOFLOW_API_KEY` and enable notebook access. Never paste the
key into a cell.

### Dataset provenance
- Plastic Waste Dataset v2 — https://universe.roboflow.com/edwin-daza-saavedra-s-workspace/plastic-waste-ag4eg (CC BY 4.0)
- COCO val2017 background images — https://cocodataset.org (CC BY 4.0), labels discarded

```bibtex
@misc{ plastic-waste-ag4eg_dataset,
  title = { Plastic Waste Dataset },
  type = { Open Source Dataset },
  author = { Edwin Daza Saavedra's Workspace },
  howpublished = { \url{ https://universe.roboflow.com/edwin-daza-saavedra-s-workspace/plastic-waste-ag4eg } },
  url = { https://universe.roboflow.com/edwin-daza-saavedra-s-workspace/plastic-waste-ag4eg },
  journal = { Roboflow Universe },
  publisher = { Roboflow },
  year = { 2026 },
  month = { jul },
  note = { visited on 2026-09-26 },
}
```

## 1. GPU, dependencies, Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "ultralytics>=8.3.0" roboflow

from google.colab import drive
drive.mount("/content/drive")  # approve the Google Drive permission prompt

import os
DRIVE = "/content/drive/MyDrive/PlasticWatch_Data"
os.makedirs(DRIVE, exist_ok=True)
print("Saving everything to", DRIVE)

## 2. Plastic Waste dataset (from Drive if saved before, else Roboflow)

In [ ]:
import os, tarfile, glob

DATA = "/content/Plastic-Waste-2"
saved = f"{DRIVE}/Plastic-Waste-2.tar"

if os.path.isdir(f"{DATA}/train/images"):
    print("Already in this runtime:", DATA)
elif os.path.exists(saved):
    print("Restoring from Drive:", saved)
    with tarfile.open(saved) as t:
        t.extractall("/content")
else:
    from google.colab import userdata
    from roboflow import Roboflow
    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    ds = (rf.workspace("edwin-daza-saavedra-s-workspace")
            .project("plastic-waste-ag4eg").version(2)
            .download("yolov11", location=DATA))
    print("Downloaded to", ds.location)

for split in ("train", "valid", "test"):
    n = len(glob.glob(f"{DATA}/{split}/images/*"))
    print(f"{split:>5}: {n} images")
!cat {DATA}/data.yaml

## 3. Background photos — scenes with no plastic

COCO val2017 photos are kept only if they contain **none** of the classes that look like
plastic litter (bottles, cups, bowls, bags, umbrellas, balls, kites, vases, toothbrushes…).
What is left is streets, parks, sky, water, rooms and animals. Labels are thrown away: each
kept photo gets an **empty** label file, which tells YOLO "there is nothing to find here".

About 15% of the training set becomes background (the Ultralytics guidance is 0–10%, a
little more here because false alarms are the problem being fixed). A separate set of
background photos is held out and never trained on — step 5 uses it to measure how often
the model raises a false alarm.

In [ ]:
import os, random, shutil, zipfile, glob, urllib.request

# COCO ids (0-based, as in the Ultralytics label files) that could look like plastic litter.
LOOKS_LIKE_PLASTIC = {
    24,  # backpack
    25,  # umbrella
    26,  # handbag
    28,  # suitcase
    29,  # frisbee
    32,  # sports ball
    33,  # kite
    39,  # bottle
    40,  # wine glass
    41,  # cup
    45,  # bowl
    75,  # vase
    79,  # toothbrush
}

def clean_scene_ids(label_dir, image_ids, exclude=LOOKS_LIKE_PLASTIC):
    '''Image ids whose label file has no excluded class (a missing file = empty scene).'''
    keep = []
    for img_id in image_ids:
        path = os.path.join(label_dir, img_id + ".txt")
        classes = set()
        if os.path.exists(path):
            with open(path) as f:
                classes = {int(line.split()[0]) for line in f if line.strip()}
        if not classes & exclude:
            keep.append(img_id)
    return keep

def add_backgrounds(ids, src_dir, dst_images, dst_labels, prefix="bg_"):
    '''Copy each photo in with an EMPTY label file: a pure negative for YOLO.'''
    os.makedirs(dst_images, exist_ok=True)
    os.makedirs(dst_labels, exist_ok=True)
    for img_id in ids:
        shutil.copy(os.path.join(src_dir, img_id + ".jpg"),
                    os.path.join(dst_images, prefix + img_id + ".jpg"))
        open(os.path.join(dst_labels, prefix + img_id + ".txt"), "w").close()
    return len(ids)

In [ ]:
COCO = "/content/coco"
os.makedirs(COCO, exist_ok=True)
if not os.path.isdir(f"{COCO}/val2017"):
    urllib.request.urlretrieve("http://images.cocodataset.org/zips/val2017.zip", f"{COCO}/val2017.zip")
    zipfile.ZipFile(f"{COCO}/val2017.zip").extractall(COCO)
if not os.path.isdir(f"{COCO}/coco/labels/val2017"):
    urllib.request.urlretrieve(
        "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco2017labels.zip",
        f"{COCO}/labels.zip")
    with zipfile.ZipFile(f"{COCO}/labels.zip") as z:
        z.extractall(COCO, [m for m in z.namelist() if "labels/val2017/" in m])

all_ids = sorted(os.path.splitext(f)[0] for f in os.listdir(f"{COCO}/val2017"))
clean = clean_scene_ids(f"{COCO}/coco/labels/val2017", all_ids)
random.Random(42).shuffle(clean)

n_train = int(0.15 * len(glob.glob(f"{DATA}/train/images/*")))
n_valid, n_heldout = 300, 400
need = n_train + n_valid + n_heldout
assert len(clean) >= need, f"only {len(clean)} clean scenes, need {need}"
tr, va, ho = clean[:n_train], clean[n_train:n_train + n_valid], clean[n_train + n_valid:need]

# Re-running must not add the backgrounds twice.
for split in ("train", "valid"):
    for f in glob.glob(f"{DATA}/{split}/*/bg_*"):
        os.remove(f)
print("clean scenes available:", len(clean), "of", len(all_ids))
print("train backgrounds:", add_backgrounds(tr, f"{COCO}/val2017", f"{DATA}/train/images", f"{DATA}/train/labels"))
print("valid backgrounds:", add_backgrounds(va, f"{COCO}/val2017", f"{DATA}/valid/images", f"{DATA}/valid/labels"))
HELDOUT = "/content/heldout_backgrounds"
shutil.rmtree(HELDOUT, ignore_errors=True)
os.makedirs(HELDOUT)
for i in ho:
    shutil.copy(f"{COCO}/val2017/{i}.jpg", HELDOUT)
print("held-out backgrounds (never trained on):", len(os.listdir(HELDOUT)))

# Stale label caches would hide the new files from YOLO.
for c in glob.glob(f"{DATA}/**/*.cache", recursive=True):
    os.remove(c)

## 4. Train YOLO11s (resumes automatically after a disconnect)

In [ ]:
import os
from ultralytics import YOLO

RUNS = f"{DRIVE}/runs"
NAME = "plastic_waste_bg_yolo11s"
last = f"{RUNS}/{NAME}/weights/last.pt"

if os.path.exists(last):
    print("Resuming from", last)
    try:
        YOLO(last).train(resume=True)
    except AssertionError as e:  # the run already finished: nothing to resume
        print(e)
else:
    YOLO("yolo11s.pt").train(
        data=f"{DATA}/data.yaml",
        epochs=60,
        imgsz=640,
        batch=16,
        patience=15,
        save_period=5,      # a checkpoint on Drive every 5 epochs
        project=RUNS,
        name=NAME,
        exist_ok=True,
        seed=42,
        deterministic=True,
    )
BEST = f"{RUNS}/{NAME}/weights/best.pt"
print("Best weights:", BEST)

## 5. Test: accuracy on plastic, and false alarms on clean scenes

Two separate questions, answered separately:

1. **Does it find plastic?** mAP50 on the dataset's own test split.
2. **Does it stay quiet when there is none?** The share of the 400 held-out background
   photos (never seen in training) on which it draws *any* box, at several confidence
   thresholds.

Put the recommended threshold in `backend/.env` as `DETECTOR_CONF_THRESHOLD`.

In [ ]:
import os
from ultralytics import YOLO

m = YOLO(BEST)
test = m.val(data=f"{DATA}/data.yaml", split="test", verbose=False)
print(f"Plastic test set  mAP50 {test.box.map50:.3f}   mAP50-95 {test.box.map:.3f}   "
      f"precision {test.box.mp:.3f}   recall {test.box.mr:.3f}")
for i, name in m.names.items():
    print(f"   {name:<16} AP50 {test.box.ap50[i]:.3f}")

print("\nFalse alarms on", len(os.listdir(HELDOUT)), "clean held-out scenes:")
preds = list(m.predict(HELDOUT, conf=0.10, imgsz=640, verbose=False, stream=True))
rows = []
for t in (0.25, 0.30, 0.35, 0.40, 0.50, 0.60):
    hit = sum(1 for p in preds if (p.boxes.conf >= t).any())
    rows.append((t, hit / len(preds)))
    print(f"   conf >= {t:.2f}: boxes on {hit:>3} photos ({100 * hit / len(preds):4.1f}%)")

ok = [t for t, rate in rows if rate <= 0.05]
pick = ok[0] if ok else rows[-1][0]
print(f"\nRecommended DETECTOR_CONF_THRESHOLD = {pick:.2f}"
      + ("" if ok else "  (no threshold reached <=5% false alarms: train longer or add backgrounds)"))

## 6. Get `best.pt` into PlasticWatch

In [ ]:
from google.colab import files
import os, shutil

shutil.copy(BEST, f"{DRIVE}/best.pt")
print(f"Also saved to Drive: {DRIVE}/best.pt  ({os.path.getsize(BEST) / 1e6:.1f} MB)")
files.download(BEST)
# Put it at plasticwatch/backend/weights/best.pt, then restart the API with DETECTOR_MODE=real.

## 7. Save the datasets to Drive

Next time, step 2 restores from these archives instead of downloading again. Each folder is
packed into one `.tar`: copying thousands of small files to Drive one by one is very slow.

In [ ]:
import os, tarfile

SOURCES = {
    "Plastic-Waste-2": DATA,
    "PlasticWatch": "/content/PlasticWatch",
    "taco": "/content/PlasticWatch/ml/data/taco",
    "yolo": "/content/PlasticWatch/ml/data/yolo",
}
for name, src in SOURCES.items():
    if not os.path.isdir(src):
        print(f"SKIP  {name}: {src} not in this runtime")
        continue
    out = f"{DRIVE}/{name}.tar"
    with tarfile.open(out, "w") as t:
        t.add(src, arcname=os.path.basename(src),
              filter=lambda ti: None if name == "PlasticWatch" and "/ml/data/" in ti.name else ti)
    print(f"SAVED {name} -> {out}")

for f in sorted(os.listdir(DRIVE)):
    p = f"{DRIVE}/{f}"
    if f.endswith(".tar"):
        with tarfile.open(p) as t:
            print(f"OK    {f}: {len(t.getnames())} entries, {os.path.getsize(p) / 1e6:.0f} MB")